# Astro Art Workshop

Neste notebook você cria uma imagem artística usando dados sintéticos inspirados em sensores astronômicos.
A demonstração segue três camadas visuais: **Sol + Universo**, **Sol + Universo + Estrelas** e **Sol + Universo + Estrelas + Gases**.

In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
from matplotlib import pyplot as plt

warnings.filterwarnings('ignore')

src_dir = Path.cwd() / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from synthetic_fits_generator import create_test_fits_collection, create_synthetic_image
from image_processor import ImageProcessor
from rgb_converter import RGBConverter
from pipeline import AstroDataVizPipeline

output_dir = Path('data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)
create_test_fits_collection(str(raw_dir))

pipeline = AstroDataVizPipeline(str(output_dir))

def show_image(rgb, title=None, figsize=(6, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(rgb, origin='lower')
    plt.axis('off')
    if title:
        plt.title(title)
    plt.show()

## 1) Sol + Universo

Neste primeiro passo, criamos uma cena com um centro brilhante e um fundo suave para representar o Sol em um ambiente cósmico.
A imagem é processada e aplicada uma paleta quente para sugerir luz solar e calor.

In [ ]:
sun_layer = create_synthetic_image((512, 512), objects=[
    {'type': 'star', 'center': (260, 260), 'brightness': 4500},
    {'type': 'nebula', 'center': (320, 320), 'radius': 80, 'brightness': 800},
])

sun_processed = ImageProcessor.apply_pipeline(
    sun_layer,
    normalize_method='percentile',
    stretch_method='asinh',
    enhance=True,
    smooth_kernel=2
)
sun_rgb = RGBConverter.colorize_monochrome(sun_processed, colormap='hot')
sun_path = output_dir / 'astro_art_sol_universo.png'
pipeline._save_rgb_image(sun_rgb, str(sun_path))
print(f'Salvo em: {sun_path}')
show_image(sun_rgb, 'Sol + Universo')

## 2) Sol + Universo + Estrelas

Agora adicionamos várias camadas de estrelas usando três bandas sintéticas.
Cada banda recebe processamento próprio e depois é combinada em RGB para revelar profundidade e estrutura.

In [ ]:
uv_layer = create_synthetic_image((512, 512), objects=[
    {'type': 'star', 'center': (120, 130), 'brightness': 700},
    {'type': 'star', 'center': (80, 420), 'brightness': 1200},
    {'type': 'star', 'center': (430, 80), 'brightness': 900},
])
blue_layer = create_synthetic_image((512, 512), objects=[
    {'type': 'star', 'center': (360, 190), 'brightness': 800},
    {'type': 'star', 'center': (280, 360), 'brightness': 650},
    {'type': 'nebula', 'center': (380, 380), 'radius': 35, 'brightness': 400},
])
green_layer = create_synthetic_image((512, 512), objects=[
    {'type': 'star', 'center': (190, 260), 'brightness': 900},
    {'type': 'star', 'center': (260, 420), 'brightness': 1100},
    {'type': 'nebula', 'center': (140, 360), 'radius': 28, 'brightness': 450},
])

uv_proc = ImageProcessor.apply_pipeline(uv_layer, normalize_method='percentile', stretch_method='asinh', enhance=True, smooth_kernel=1)
blue_proc = ImageProcessor.apply_pipeline(blue_layer, normalize_method='percentile', stretch_method='asinh', enhance=True, smooth_kernel=1)
green_proc = ImageProcessor.apply_pipeline(green_layer, normalize_method='percentile', stretch_method='asinh', enhance=True, smooth_kernel=1)

rgb_stars = RGBConverter.combine_channels(uv_proc, blue_proc, green_proc, weights=(0.9, 1.0, 1.1))
stars_path = output_dir / 'astro_art_sol_estrelas.png'
pipeline._save_rgb_image(rgb_stars, str(stars_path))
print(f'Salvo em: {stars_path}')
show_image(rgb_stars, 'Sol + Universo + Estrelas')

## 3) Sol + Universo + Estrelas + Gases

A terceira camada adiciona uma composição de gases com cores frias.
O resultado final é uma mescla entre os canais de estrelas e as estruturas gasosas.

In [ ]:
gas_layer = create_synthetic_image((512, 512), objects=[
    {'type': 'nebula', 'center': (200, 280), 'radius': 70, 'brightness': 1000},
    {'type': 'nebula', 'center': (340, 180), 'radius': 55, 'brightness': 900},
    {'type': 'galaxy', 'center': (260, 120), 'radius': 50, 'brightness': 700},
])
gas_processed = ImageProcessor.apply_pipeline(
    gas_layer,
    normalize_method='percentile',
    stretch_method='sqrt',
    enhance=True,
    smooth_kernel=2
)
gas_rgb = RGBConverter.colorize_monochrome(gas_processed, colormap='cool')
final_art = np.clip(0.65 * rgb_stars + 0.35 * gas_rgb, 0, 1)
final_path = output_dir / 'astro_art_sol_estrelas_gases.png'
pipeline._save_rgb_image(final_art, str(final_path))
print(f'Salvo em: {final_path}')
show_image(final_art, 'Sol + Universo + Estrelas + Gases')

## Imagens prontas para impressão

As imagens finais foram salvas em `data/processed/`.
Use essas PNGs para criar quadros com a composição visual que você deseja.